# Face model v2 — round 2: noise check + combined recipe
Round 1 (val Macro-F1, v1 = 0.6566): res112 0.6634, mix_e30 0.6606, cw075 0.6422, mix05 0.6417 — all within about ±0.015, one seed each.

This round:
- **v2_seed42 / v2_seed1 / v2_seed2** = the v1 recipe, same validation split (`--split-seed 42`), only the training seed changes → how much Macro-F1 moves by luck alone.
- **v2_combo** = CutMix 0.5 + 30 epochs + class-weight 0.75 at res 96 (no extra latency).

Needs in `MyDrive/face-v2/`: `Face-Lab-code-r2.zip` (new: adds `--split-seed`), the FER2013 zip, and `out/face_v2_results.zip` from round 1. Runtime → T4 GPU.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile
D='/content/drive/MyDrive/face-v2'
os.makedirs('/content/Main_demo', exist_ok=True)
zipfile.ZipFile(f'{D}/Face-Lab-code-r2.zip').extractall('/content/Main_demo')
fer = [p for p in glob.glob(f'{D}/*.zip') if 'Face-Lab' not in p][0]
zipfile.ZipFile(fer).extractall('/content/Main_demo/FER2013-Data')
zipfile.ZipFile(f'{D}/out/face_v2_results.zip').extractall('/content/Main_demo/Face-Lab')   # round-1 results/ for the table
for s in ('train','test'):
    print(s, sum(len(f) for _,_,f in os.walk(f'/content/Main_demo/FER2013-Data/{s}')))   # expect 28709 / 7178
%cd /content/Main_demo/Face-Lab
print(sorted(os.listdir('results')))

In [ ]:
BASE = '--arch efficientnet_b0 --res 96 --epochs 20 --head-epochs 1 --unfreeze 0.0 --split-seed 42'
EXPS = {
  'v2_seed42': '--seed 42',
  'v2_seed1':  '--seed 1',
  'v2_seed2':  '--seed 2',
  'v2_combo':  '--seed 42 --mix 0.5 --epochs 30 --patience 8 --cw-power 0.75',
}
for tag, extra in EXPS.items():
    cmd = f'python train.py {BASE} {extra} --tag {tag}'
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else '', r.stderr[-600:] if r.returncode else '', flush=True)

In [ ]:
import json, glob, statistics as st
rows={}
for f in sorted(glob.glob('results/v2_*.json')):
    r=json.load(open(f)); pc=r['val']['per_class']
    rows[r['tag']]=(r['best_val_macro_f1'], r['val']['accuracy'], pc['sad']['f1'], pc['fear']['f1'], r['best_epoch'], r['train_minutes'])
seeds=[rows[t][0] for t in ('v2_seed42','v2_seed1','v2_seed2') if t in rows]
if len(seeds)>1:
    m, s = st.mean(seeds), st.stdev(seeds)
    print(f'v1 recipe over {len(seeds)} seeds: mean {m:.4f}  sd {s:.4f}  range {min(seeds):.4f}-{max(seeds):.4f}')
    print(f'=> a run beats v1 only if it is clearly above {max(seeds):.4f} (ideally > mean + 2 sd = {m+2*s:.4f})')
    print()
print(f"{'run':12s} {'macroF1':>8s} {'acc':>6s} {'sad':>6s} {'fear':>6s} {'ep':>3s} {'min':>5s}")
for t,(mf,a,sa,fe,ep,mn) in sorted(rows.items(), key=lambda kv:-kv[1][0]):
    print(f'{t:12s} {mf:8.4f} {a:6.3f} {sa:6.3f} {fe:6.3f} {ep:3d} {mn:5.1f}')

In [ ]:
OUT='/content/drive/MyDrive/face-v2/out'
shutil.make_archive('/content/face_v2_results_r2','zip','/content/Main_demo/Face-Lab','results')
shutil.copy('/content/face_v2_results_r2.zip', OUT)
for t in EXPS:
    if os.path.exists(f'runs/{t}/best.pt'):
        os.makedirs(f'{OUT}/{t}', exist_ok=True); shutil.copy(f'runs/{t}/best.pt', f'{OUT}/{t}/best.pt')
print(sorted(os.listdir(OUT)))